# 03 — Analysis & Visualizations
**Goal:** Answer key questions about Medicaid spending using the exported Parquet files.

**Key Questions:**
1. How has total Medicaid spending trended 2018–2024?
2. Which states spend the most per beneficiary?
3. What are the top 20 costliest HCPCS procedures?
4. Who are the top 50 providers by total paid?
5. Which providers have anomalously high billing rates? (fraud signals)

In [1]:
import duckdb
import polars as pl
import plotly.express as px
import matplotlib.pyplot as plt

con = duckdb.connect()

# Load from pre-exported Parquet files (fast, no joins needed)
by_state = con.sql(
    "SELECT * FROM read_parquet('../exports/by_state_month.parquet')"
).pl()
by_hcpcs = con.sql(
    "SELECT * FROM read_parquet('../exports/by_hcpcs_month.parquet')"
).pl()
providers = con.sql(
    "SELECT * FROM read_parquet('../exports/top_providers.parquet')"
).pl()

print("Loaded all exports successfully")

IOException: IO Error: No files found that match the pattern "../exports/by_state_month.parquet"

## 1. Spending Trend Over Time

In [ ]:
df_time = (
    by_state.group_by("year", "month")
    .agg(pl.col("total_paid").sum())
    .sort(["year", "month"])
    .with_columns((pl.col("year") + "-" + pl.col("month")).alias("period"))
    .to_pandas()
)

fig = px.line(
    df_time,
    x="period",
    y="total_paid",
    title="Total Medicaid Spending Over Time",
    labels={"total_paid": "Total Paid ($)", "period": "Month"},
)
fig.show()

## 2. Spending by State (Total)

In [ ]:
df_state = (
    by_state.group_by("state")
    .agg(pl.col("total_paid").sum(), pl.col("total_beneficiaries").sum())
    .with_columns(
        (pl.col("total_paid") / pl.col("total_beneficiaries")).alias(
            "paid_per_beneficiary"
        )
    )
    .sort("total_paid", descending=True)
    .to_pandas()
)

fig = px.choropleth(
    df_state,
    locations="state",
    locationmode="USA-states",
    color="total_paid",
    scope="usa",
    title="Total Medicaid Spending by State",
    color_continuous_scale="Blues",
)
fig.show()

## 3. Top 20 Costliest HCPCS Procedures

In [ ]:
df_hcpcs = (
    by_hcpcs.group_by("HCPCS_CODE", "hcpcs_description")
    .agg(pl.col("total_paid").sum())
    .sort("total_paid", descending=True)
    .head(20)
    .to_pandas()
)

fig = px.bar(
    df_hcpcs,
    x="total_paid",
    y="HCPCS_CODE",
    orientation="h",
    hover_data=["hcpcs_description"],
    title="Top 20 HCPCS Procedures by Total Spend",
    labels={"total_paid": "Total Paid ($)", "HCPCS_CODE": "Code"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 4. Top 50 Providers by Total Paid

In [ ]:
top50 = providers.head(50).to_pandas()

fig = px.scatter(
    top50,
    x="total_claims",
    y="total_paid",
    hover_data=["npi", "last_name", "state"],
    size="total_beneficiaries",
    color="state",
    title="Top 50 Providers: Claims vs Total Paid",
    labels={"total_claims": "Total Claims", "total_paid": "Total Paid ($)"},
)
fig.show()

## 5. Anomaly Detection — High Billing Flags
Flag providers whose `avg_paid_per_claim` is **3x above the national average** for their HCPCS code.

In [ ]:
national_avg = by_hcpcs.group_by("HCPCS_CODE").agg(
    pl.col("avg_paid_per_claim").mean().alias("national_avg")
)

# TODO: Join with top_providers on HCPCS_CODE and flag where
# provider avg_paid_per_claim > 3 * national_avg
# Export to '../exports/anomaly_flags.parquet'